# The distilled vitals student (slice M1): training, from the run log

A larger teacher network is trained on the true labels; small student networks are trained to copy the teacher's
logits (distilled) or the labels (hard-label, the reference). This notebook only reads the run log
(`training/runs/student_vitals/`), `metadata.json` and the file sizes of the model folder, and draws them. It trains
nothing. **Synthetic, not validated.**

In [ ]:
# Finds the repository root (the folder that holds models/ and training/). This notebook only reads and draws.
import json
import logging
import re
from pathlib import Path

logging.getLogger("matplotlib").setLevel(logging.ERROR)
%matplotlib inline
import matplotlib.pyplot as plt
from IPython.display import Markdown, display
from matplotlib.ticker import MaxNLocator

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "models").is_dir() and (p / "training").is_dir())
COLORS = ["#0072B2", "#D55E00", "#009E73", "#CC79A7", "#E69F00", "#56B4E9", "#000000"]  # Okabe-Ito
plt.rcParams.update({"figure.dpi": 100, "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True,
                     "grid.alpha": 0.25, "font.size": 9})


def ints(*axes):
    """Whole numbers on the x axis (generations and epochs)."""
    for ax in axes:
        ax.xaxis.set_major_locator(MaxNLocator(integer=True))


def newest_log(model):
    """The run log with the newest file name under training/runs/<model>/ (the names carry the date)."""
    logs = sorted((ROOT / "training" / "runs" / model).glob("*.json"))
    return json.loads(logs[-1].read_text()), logs[-1].name


In [ ]:
log, log_name = newest_log("student_vitals")
SV = ROOT / "models" / "tier_c" / "student_vitals"
meta = json.loads((SV / "metadata.json").read_text())
TAGS = [t for t in ("all", "no_temp") if t in log["variants"]]
LABEL = {"all": "with temperature", "no_temp": "without temperature"}
print(f"run log {log_name}: run id {log['run_id']}, commit {log['commit']}")
print(f"{log['framework']} {log['framework_version']}; train seeds {log['data']['train_seeds'][0]}-{log['data']['train_seeds'][-1]} "
      f"({len(log['data']['train_seeds'])} trials), validation seeds {log['data']['validation_seeds'][0]}-{log['data']['validation_seeds'][-1]} "
      f"({len(log['data']['validation_seeds'])} trials); budget cuts: {log['budget_cuts'] or 'none'}")
for t in TAGS:
    d = log["data"]["variants"][t]
    print(f"{LABEL[t]}: {d['training_rows']} training rows ({d['training_positives']} positive), {d['validation_rows']} validation rows ({d['validation_positives']} positive); shipped: {log['variants'][t]['chosen']['shipped']}")

## Loss per epoch

Top row: the mean training loss of every fit. Bottom row: its validation log loss. Distilled students are trained on the teacher's logits, so their training loss is a mean squared error, not a log loss; the validation log loss is comparable across all fits. A ring marks the best epoch (the one kept).

In [ ]:
GROUPS = [("two teachers", "teacher", "training loss (log loss, true labels)"),
          ("six distilled students", "distilled", "training loss (MSE on teacher logits)"),
          ("six hard-label students", "hard_label", "training loss (log loss, true labels)")]


def short(f):
    return f"decay {f['weight_decay']:g}" if f["kind"] == "teacher" else f"{f['inputs']} w{f['width']}"


for tag in TAGS:
    fig, axes = plt.subplots(2, 3, figsize=(13, 6.2))
    for j, (title, kind, ylab) in enumerate(GROUPS):
        for f, c in zip([f for f in log["variants"][tag]["fits"] if f["kind"] == kind], COLORS * 2):
            e = list(range(1, len(f["train_loss_by_epoch"]) + 1))
            axes[0, j].plot(e, f["train_loss_by_epoch"], "-o", ms=3, color=c, label=short(f))
            axes[1, j].plot(e, f["val_log_loss_by_epoch"], "-o", ms=3, color=c)
            axes[1, j].plot(f["best_epoch"], f["val_log_loss_by_epoch"][f["best_epoch"] - 1], "o", ms=10, mfc="none", mec=c)
        axes[0, j].set_title(title)
        axes[0, j].set_ylabel(ylab)
        axes[1, j].set_ylabel("validation log loss")
        axes[1, j].set_xlabel("epoch")
        axes[0, j].legend(fontsize=7)
        ints(axes[0, j], axes[1, j])
    fig.suptitle(f"Loss per epoch, {LABEL[tag]}")
    fig.tight_layout()
    plt.show()
    fits = log["variants"][tag]["fits"]
    best = min(fits, key=lambda f: f["final_val_log_loss"])
    print(f"Reading ({LABEL[tag]}): {len(fits)} fits; the lowest validation log loss is {best['final_val_log_loss']} ({best['name']}, best epoch {best['best_epoch']}).")

## Validation log loss and average precision

The teacher, the chosen student, the best hard-label student and the references: the constant (training positive rate), gradient boosting on the student's base inputs, and regularised gradient boosting. Lower log loss is better; higher average precision is better. The references have no average precision here.

In [ ]:
def fit_of(tag, kind, **want):
    return next(f for f in log["variants"][tag]["fits"] if f["kind"] == kind and all(f[k] == v for k, v in want.items()))


def scores(tag):
    v, m = log["variants"][tag], meta["variants"][tag]
    out = {"teacher": fit_of(tag, "teacher", weight_decay=v["chosen"]["teacher"]["weight_decay"])}
    s = v["chosen"]["student"]
    if s is not None:
        out["chosen student"] = fit_of(tag, "distilled" if s["loss"] == "mse" else "hard_label", inputs=s["inputs"], width=s["width"])
    h = m["references"]["hard_label_mlp"]
    out["best hard-label student"] = fit_of(tag, "hard_label", inputs=h["inputs"], width=h["width"])
    loss = {k: f["final_val_log_loss"] for k, f in out.items()}
    ap = {k: f["average_precision"] for k, f in out.items()}
    refs = v["references"]
    loss["constant"] = refs["constant"]["val_log_loss"]
    loss["boosting, base inputs"] = refs["old_recipe"]["val_log_loss"]
    if refs.get("regularised_gradient_boosting"):
        loss["regularised boosting"] = refs["regularised_gradient_boosting"]["val_log_loss"]
    return loss, ap


fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
names = ["teacher", "chosen student", "best hard-label student", "constant", "boosting, base inputs", "regularised boosting"]
allscores = {t: scores(t) for t in TAGS}
for ax, which, title, ylab in ((axes[0], 0, "Validation log loss (lower is better)", "log loss"), (axes[1], 1, "Validation average precision (higher is better)", "average precision")):
    w = 0.8 / len(TAGS)
    for i, (tag, c) in enumerate(zip(TAGS, COLORS)):
        d = allscores[tag][which]
        xs = [k for k, n in enumerate(names) if n in d]
        ax.bar([k + (i - (len(TAGS) - 1) / 2) * w for k in xs], [d[names[k]] for k in xs], width=w, color=c, label=LABEL[tag])
    ax.set_xticks(range(len(names)), names, rotation=25, ha="right")
    ax.set_title(title)
    ax.set_ylabel(ylab)
    ax.legend()
fig.tight_layout()
plt.show()
for tag in TAGS:
    loss, ap = allscores[tag]
    print(f"Reading ({LABEL[tag]}): log loss {loss}; average precision {ap}.")

## The teacher gate

Paired difference in validation log loss, teacher minus reference, with the 95% patient-level bootstrap interval. Below zero means the teacher is better; the gate passes when the upper end of the interval is below zero (filled marker). The teacher must also be at least 10 times larger than the student.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.6))
y, ticks = 0, []
for tag, c in zip(TAGS, COLORS):
    gate = log["variants"][tag]["teacher_gate"]
    for ref in ("constant", "old_recipe", "hard_label_mlp"):
        g = gate[ref]
        ax.errorbar(g["difference"], y, xerr=[[g["difference"] - g["lower_2_5"]], [g["upper_97_5"] - g["difference"]]], fmt="o", color=c,
                    mfc=c if g["passes"] else "none", capsize=3)
        ticks.append((y, f"{LABEL[tag]}: vs {ref.replace('_', ' ')}"))
        y += 1
ax.axvline(0, color="black", lw=0.8)
ax.set_yticks([t[0] for t in ticks], [t[1] for t in ticks])
ax.invert_yaxis()
ax.set_title("Teacher gate: validation log loss, teacher minus reference")
ax.set_xlabel("difference in log loss (negative: teacher better)")
plt.show()
for tag in TAGS:
    gate = log["variants"][tag]["teacher_gate"]
    print(f"Reading ({LABEL[tag]}): " + "; ".join(f"vs {k}: {v['difference']} ({v['lower_2_5']} to {v['upper_97_5']}), passes {v['passes']}" for k, v in gate.items() if isinstance(v, dict))
          + f"; teacher much bigger: {gate['much_bigger']}; gate passes: {gate['passes']}.")

## The distillation gap

For each of the six distilled students: its validation log loss minus the teacher's, and its fidelity (root mean squared difference from the teacher's log-odds on validation rows). A ring marks the chosen student when it is distilled.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, key, title, ylab in ((axes[0], "log_loss_minus_teacher", "Distillation gap", "student minus teacher, validation log loss"),
                            (axes[1], "fidelity_rmse_log_odds", "Fidelity to the teacher", "RMSE of log-odds")):
    w = 0.8 / len(TAGS)
    for i, (tag, c) in enumerate(zip(TAGS, COLORS)):
        grid = meta["variants"][tag]["grid"]["distilled"]
        chosen = log["variants"][tag]["chosen"]["student"]
        for k, s in enumerate(grid):
            pick = chosen is not None and chosen["loss"] == "mse" and (s["inputs"], s["width"]) == (chosen["inputs"], chosen["width"])
            ax.bar(k + (i - (len(TAGS) - 1) / 2) * w, s[key], width=w, color=c, label=LABEL[tag] if k == 0 else None,
                   edgecolor="black" if pick else "none", linewidth=1.8)
    ax.set_xticks(range(len(grid)), [f"{s['inputs']} w{s['width']}" for s in grid], rotation=25, ha="right")
    ax.set_title(title)
    ax.set_ylabel(ylab)
    ax.legend()
fig.tight_layout()
plt.show()
for tag in TAGS:
    s = log["variants"][tag]["chosen"]["student"]
    t = log["variants"][tag]["chosen"]["teacher"]
    if s is None:
        print(f"Reading ({LABEL[tag]}): no student was shipped.")
    else:
        print(f"Reading ({LABEL[tag]}): chosen student ({s['inputs']}, width {s['width']}, {'distilled' if s['loss'] == 'mse' else 'hard-label'}) "
              f"has validation log loss {s['val_log_loss']}, teacher {t['val_log_loss']}, difference {round(s['val_log_loss'] - t['val_log_loss'], 6)}.")

## Parameter counts and file sizes

The teacher is the large network; the student is what ships. The ratio is the teacher's parameter count over the student's.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.6))
w = 0.8 / len(TAGS)
for i, (tag, c) in enumerate(zip(TAGS, COLORS)):
    p = log["variants"][tag]["parameters"]
    ax.bar([k + (i - (len(TAGS) - 1) / 2) * w for k in (0, 1)], [p["teacher"], p["student"] or 0], width=w, color=c, label=LABEL[tag])
ax.set_yscale("log")
ax.set_xticks([0, 1], ["teacher", "chosen student"])
ax.set_title("Parameter counts")
ax.set_ylabel("parameters (log scale)")
ax.legend()
plt.show()
for tag in TAGS:
    p = log["variants"][tag]["parameters"]
    print(f"Reading ({LABEL[tag]}): teacher {p['teacher']} parameters, student {p['student']}, ratio {p['ratio_teacher_to_student']}.")
rows = ["| file | bytes |", "|---|---|"]
for f in sorted(SV.iterdir()):
    if f.is_file() and f.suffix not in (".md", ".sh", ".txt"):
        rows.append(f"| {f.name} | {f.stat().st_size} |")
display(Markdown("\n".join(rows)))